# Compute cycling access and egress travel time matrix
Zehui Yin

In [ ]:
import datetime
from pathlib import Path

import geopandas as gpd
import pandas as pd
import r5py

In [ ]:
# build the transport network
transport_network = r5py.TransportNetwork(
    "data/cropped-ontario-260320.osm.pbf",
    [
        "data/HSR_Transit_Feed.zip",
        "data/TTC_Routes_and_Schedules_Data.zip",
    ]
)

In [ ]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

# read in hospital opportunities
hospital_opportunities = gpd.read_file("output/hospital_opportunities.geojson")
hospital_opportunities["id"] = hospital_opportunities["opportunity_id"]

# read in analysis cells
analysis_cells = gpd.read_file("output/analysis_grid_h3_res9.geojson")
analysis_cells = analysis_cells.to_crs(epsg=26917)
analysis_cells["id"] = analysis_cells["h3_id"]

# convert hexagons to centroid points
analysis_points = analysis_cells.copy()
analysis_points["geometry"] = analysis_points.geometry.centroid

In [ ]:
SCENARIOS = {
    'weekday_peak_cycle': datetime.datetime(2026, 3, 17, 8, 0, 0),
    'weekday_off_peak_cycle': datetime.datetime(2026, 3, 17, 14, 0, 0),
    'weekend_peak_cycle': datetime.datetime(2026, 3, 22, 13, 0, 0),
    'weekend_off_peak_cycle': datetime.datetime(2026, 3, 22, 20, 0, 0),
}

SCENARIO_OUTPUT_FILES = {
    scenario_name: OUTPUT_DIR / f'transit_ttm_{scenario_name}.parquet'
    for scenario_name in SCENARIOS
}

travel_time_matrices = {}

for scenario_name, departure in SCENARIOS.items():
    travel_time_matrix = r5py.TravelTimeMatrix(
        transport_network,
        origins=analysis_points,
        destinations=hospital_opportunities,
        transport_modes=[r5py.TransportMode.TRANSIT, r5py.TransportMode.BICYCLE],
        departure=departure,
        access_modes=[r5py.TransportMode.BICYCLE],
        egress_modes=[r5py.TransportMode.BICYCLE],
        max_time_cycling=datetime.timedelta(minutes=30)
    ) # type: ignore
    output_path = SCENARIO_OUTPUT_FILES[scenario_name]
    travel_time_matrix.to_parquet(output_path, index=False)
    travel_time_matrices[scenario_name] = travel_time_matrix

In [ ]:
pd.DataFrame(
    [
        {
            "scenario": scenario_name,
            "departure": SCENARIOS[scenario_name],
            "rows": len(travel_time_matrices[scenario_name]),
            "output_file": str(SCENARIO_OUTPUT_FILES[scenario_name]),
        }
        for scenario_name in SCENARIOS
    ]
)